# Objectives

### English

Re-run the current national team roster matching process using the expanded International Player Database V02.

The main objectives of this notebook are:

- Match current national team players against `international_player_database_v02.csv`.
- Measure overall player coverage.
- Analyze coverage by national team.
- Compare the new results against Current Roster Matching V01.
- Quantify the impact of incorporating club competition histories.
- Generate the final matched roster dataset required to build current team vectors.

### Español

Reejecutar el proceso de matching de los planteles actuales utilizando la versión expandida de la International Player Database V02.

Los principales objetivos de esta notebook son:

- Vincular los jugadores actuales de cada selección con `international_player_database_v02.csv`.
- Medir la cobertura general de jugadores.
- Analizar la cobertura por selección nacional.
- Comparar los nuevos resultados con Current Roster Matching V01.
- Cuantificar el impacto de incorporar historiales de competiciones de clubes.
- Generar el dataset final de planteles vinculados necesario para construir los vectores actuales de selección.

# Configuration

## Imports

In [ ]:
from pathlib import Path
import sys
import unicodedata
import re

import numpy as np
import pandas as pd

## Custom Pd Options

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

## Paths

### Base

In [ ]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RAW_DIR = DATA_DIR / "raw"

PROJECT_ROOT

WindowsPath('..')

### Initial CSVs

In [ ]:
CURRENT_ROSTERS_PATH = (
    RAW_DIR / "current_rosters" / "current_rosters_raw_v01.csv"
)

PLAYER_DATABASE_V02_PATH = (
    PROCESSED_DIR / "international_player_database_v02.csv"
)

NAME_CORRECTIONS_PATH = (
    PROCESSED_DIR / "name_corrections_v01.csv"
)

MATCHING_V01_PATH = (
    PROCESSED_DIR / "current_roster_matching_v01.csv"
)

### Output CSVs

In [ ]:
OUTPUT_MATCHING_PATH = (
    PROCESSED_DIR / "current_roster_matching_v02.csv"
)

OUTPUT_UNMATCHED_PATH = (
    PROCESSED_DIR / "unmatched_roster_players_v02.csv"
)

OUTPUT_COVERAGE_PATH = (
    PROCESSED_DIR / "roster_coverage_report_v02.csv"
)

OUTPUT_COMPARISON_PATH = (
    PROCESSED_DIR / "current_roster_matching_comparison_v01_v02.csv"
)

### Checks

In [ ]:
required_paths = {
    "Current rosters": CURRENT_ROSTERS_PATH,
    "International Player Database V02": PLAYER_DATABASE_V02_PATH,
    "Name corrections V01": NAME_CORRECTIONS_PATH,
    "Current roster matching V01": MATCHING_V01_PATH,
}

for file_name, file_path in required_paths.items():
    print(f"{file_name}:")
    print(f"  Path: {file_path}")
    print(f"  Exists: {file_path.exists()}")

Current rosters:
  Path: ..\data\raw\current_rosters\current_rosters_raw_v01.csv
  Exists: True
International Player Database V02:
  Path: ..\data\processed\international_player_database_v02.csv
  Exists: True
Name corrections V01:
  Path: ..\data\processed\name_corrections_v01.csv
  Exists: True
Current roster matching V01:
  Path: ..\data\processed\current_roster_matching_v01.csv
  Exists: True


In [ ]:
missing_paths = [
    file_path
    for file_path in required_paths.values()
    if not file_path.exists()
]

assert not missing_paths, (
    "The following required files were not found:\n"
    + "\n".join(str(path) for path in missing_paths)
)

print("All required input files were found.")

All required input files were found.


## Load Datasets

In [ ]:
current_rosters = pd.read_csv(CURRENT_ROSTERS_PATH)

player_database_v02 = pd.read_csv(
    PLAYER_DATABASE_V02_PATH
)

name_corrections = pd.read_csv(
    NAME_CORRECTIONS_PATH
)

matching_v01 = pd.read_csv(
    MATCHING_V01_PATH
)

In [ ]:
print("Current rosters:", current_rosters.shape)
print("International Player Database V02:", player_database_v02.shape)
print("Name corrections:", name_corrections.shape)
print("Current roster matching V01:", matching_v01.shape)

Current rosters: (1248, 4)
International Player Database V02: (4058, 33)
Name corrections: (75, 7)
Current roster matching V01: (1248, 10)


### Display

In [ ]:
print("current_rosters")
display(current_rosters.head())
print("\n")

print("player_database_v02")
display(player_database_v02.head())
print("\n")

print("name_corrections")
display(name_corrections.head())
print("\n")

print("matching_v01")
display(matching_v01.head())
print("\n")


current_rosters


,country,player_name,position,shirt_number
0,Algeria (ALG),MASTIL Melvin,Goalkeeper,1
1,Algeria (ALG),MANDI Aissa,Defender,2
2,Algeria (ALG),ABADA Achref,Defender,3
3,Algeria (ALG),TOUGAI Mohamed,Defender,4
4,Algeria (ALG),BELAID Zineddine,Defender,5




player_database_v02


,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,Dispossessed,Dribble,Dribbled Past,Duel,Foul Committed,Foul Won,Goal Keeper,Interception,Miscontrol,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On,Own Goal For
0,2935,Nordi Mukiele Mulere,Paris Saint-Germain,0.0,0.0,0.0,3.0,0.0,399,26,16,344,14,10,8,7,25,14,7,0,13,4,455,2.0,2.0,107,4.0,5,0.0,0.0,0.0,0.0,0.0
1,2936,Christophe Kerbrat,Guingamp,0.0,0.0,0.0,0.0,0.0,12,3,2,13,7,0,0,3,6,2,1,0,2,0,28,0.0,0.0,18,0.0,0,0.0,0.0,0.0,0.0,0.0
2,2937,Christ-Emmanuel Faitout Maouassa,Montpellier,0.0,0.0,0.0,0.0,0.0,50,5,1,42,0,1,5,4,3,3,4,0,3,2,44,0.0,0.0,27,0.0,0,0.0,0.0,0.0,0.0,0.0
3,2940,Jean Eudès Aholou,Strasbourg,0.0,0.0,0.0,0.0,0.0,9,1,0,9,0,0,0,0,0,2,1,0,2,0,9,0.0,0.0,4,0.0,0,0.0,0.0,0.0,0.0,0.0
4,2941,Ismaïla Sarr,Senegal,11.0,2.0,3.0,5.0,0.0,547,39,8,384,8,27,42,9,30,14,56,0,5,36,296,2.0,2.0,250,0.0,30,0.0,1.0,0.0,0.0,0.0




name_corrections


,country,roster_name,roster_name_normalized,database_name,database_name_normalized,correction_type,notes
0,Algeria (ALG),BOUDAOUI Hicham,boudaoui hicham,Hichem Boudaoui,hichem boudaoui,spelling/accent,manually proposed from roster/database country...
1,Argentina (ARG),GONZALEZ Nico,gonzalez nico,Nicolás Iván González,nicolas ivan gonzalez,nickname/full_name,manually proposed from roster/database country...
2,Austria (AUT),MWENE Phillip,mwene phillip,Philipp Mwene,philipp mwene,spelling,manually proposed from roster/database country...
3,Austria (AUT),SCHOEPF Alessandro,schoepf alessandro,Alessandro Schöpf,alessandro schopf,accent/transliteration,manually proposed from roster/database country...
4,Brazil (BRA),MARQUINHOS Marcos,marquinhos marcos,Marcos Aoás Corrêa,marcos aoas correa,nickname/full_name,manually proposed from roster/database country...




matching_v01


,country,country_clean,roster_name,position,shirt_number,player_id,database_name,database_team,matched,match_method
0,Algeria (ALG),Algeria,MASTIL Melvin,Goalkeeper,1,NaN,NaN,NaN,False,unmatched
1,Algeria (ALG),Algeria,MANDI Aissa,Defender,2,6648.0,Aïssa Mandi,Algeria,True,token_subset_country
2,Algeria (ALG),Algeria,ABADA Achref,Defender,3,NaN,NaN,NaN,False,unmatched
3,Algeria (ALG),Algeria,TOUGAI Mohamed,Defender,4,160943.0,Mohamed Amine Tougai,Algeria,True,token_subset_country
4,Algeria (ALG),Algeria,BELAID Zineddine,Defender,5,NaN,NaN,NaN,False,unmatched


### Columns

In [ ]:
datasets = {
    "Current rosters": current_rosters,
    "Player database V02": player_database_v02,
    "Name corrections": name_corrections,
    "Matching V01": matching_v01,
}

for dataset_name, dataframe in datasets.items():
    print(f"\n{dataset_name} columns:")
    print(dataframe.columns.tolist())


Current rosters columns:
['country', 'player_name', 'position', 'shirt_number']

Player database V02 columns:
['player_id', 'player_name', 'team_name', 'matches_played', 'competitions_played', 'seasons_played', '50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Error', 'Offside', 'Own Goal Against', 'Camera On', 'Own Goal For']

Name corrections columns:
['country', 'roster_name', 'roster_name_normalized', 'database_name', 'database_name_normalized', 'correction_type', 'notes']

Matching V01 columns:
['country', 'country_clean', 'roster_name', 'position', 'shirt_number', 'player_id', 'database_name', 'database_team', 'matched', 'match_method']


### Checks

In [ ]:
assert not current_rosters.empty
assert not player_database_v02.empty
assert not matching_v01.empty

assert "player_id" in player_database_v02.columns
assert "player_name" in player_database_v02.columns
assert "team_name" in player_database_v02.columns

assert player_database_v02["player_id"].is_unique, (
    "player_id must be unique in International Player Database V02."
)

print("Initial structural checks passed.")

Initial structural checks passed.


## Utility Functions

In [ ]:
def normalize_name(name):
    """
    Normalize player names for matching.

    - Converts to lowercase.
    - Removes accents.
    - Removes punctuation.
    - Normalizes whitespace.
    """
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    name = unicodedata.normalize("NFKD", name)
    name = "".join(char for char in name if not unicodedata.combining(char))

    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name


In [ ]:
def clean_roster_country(country):
    return re.sub(r"\s*\([A-Z]{3}\)$", "", country).strip()

In [ ]:
def build_name_lookup(player_database):
    """
    Build a normalized player-name lookup and record how many database
    players share each normalized name.
    """
    lookup = player_database.copy()

    lookup["database_name_normalized"] = (
        lookup["player_name"]
        .fillna("")
        .map(normalize_name)
    )

    lookup["normalized_name_frequency"] = (
        lookup.groupby("database_name_normalized")["player_id"]
        .transform("nunique")
    )

    return lookup